In [2]:
import pandas as pd
import requests
import time
from tqdm import tqdm

In [4]:
# 1. Ler planilha
df = pd.read_excel("Dados-iniciais/todos03.xlsx")

In [5]:
# 2. Garantir que coordenadas sejam numéricas
coord_cols = ["LAT_MUNI", "LON_MUNI", "LAT_PESA", "LON_PESA"]

for col in coord_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .astype(float)
    )

In [6]:

# 3. Função OSRM
def calcular_rota_osrm(lat_origem, lon_origem, lat_destino, lon_destino):
    url = (
        f"https://router.project-osrm.org/route/v1/driving/"
        f"{lon_origem},{lat_origem};{lon_destino},{lat_destino}"
        f"?overview=false"
    )
    
    try:
        r = requests.get(url, timeout=30)
        data = r.json()
        
        if data.get("code") == "Ok":
            rota = data["routes"][0]
            distancia_km = rota["distance"] / 1000
            tempo_min = rota["duration"] / 60
            return distancia_km, tempo_min, "ok"
        else:
            return None, None, data.get("code")
    
    except Exception as e:
        return None, None, str(e)

In [7]:
# 4. Calcular rotas
distancias = []
tempos = []
status = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    dist_km, tempo_min, st = calcular_rota_osrm(
        row["LAT_MUNI"],
        row["LON_MUNI"],
        row["LAT_PESA"],
        row["LON_PESA"]
    )
    
    distancias.append(dist_km)
    tempos.append(tempo_min)
    status.append(st)
    
    time.sleep(1)  # respeita o servidor público do OSRM

100%|██████████| 491/491 [18:58<00:00,  2.32s/it]


In [8]:
# 5. Adicionar resultados
df["DIST_OSRM_KM"] = distancias
df["TEMPO_OSRM_MIN"] = tempos
df["STATUS_OSRM"] = status

In [9]:
# 6. Arredondar
df["DIST_OSRM_KM"] = df["DIST_OSRM_KM"].round(1)
df["TEMPO_OSRM_MIN"] = df["TEMPO_OSRM_MIN"].round(1)

In [10]:
# 7. Salvar resultado bruto
df.to_excel("todos03_com_rotas_osrm.xlsx", index=False)

In [11]:
# 8. Multiplo Pesa / menor rota
df_menor_tempo = (
    df.sort_values("TEMPO_OSRM_MIN")
      .groupby("REFERENCIADO", as_index=False)
      .first()
)

df_menor_tempo.to_excel("todos03_rotas_menor_tempo.xlsx", index=False)

KeyError: 'REFERENCIADO'